# Dataset Tarea B con WHISKiT Physics: forma + pose del objeto a partir del torque

Genera simulaciones con tres familias de objetos (cilindro, prisma, pared ondulada; 3 tamaños cada una) en poses aleatorias, con dos garantías por construcción: el objeto **no atraviesa** ningún bigote en reposo (evita el artefacto de penetración) y **sí es alcanzado** por algún bigote durante el batido (toda muestra tiene contacto).

Etiquetas por simulación: `shape_family`, `size_param`, `mesh`, `x_mm`, `y_mm`, `yaw_deg`.

Pasos: montar Drive → compilar el simulador con los parches (`--PEG_LOC`, `--ENV_LOC/ENV_YAW/ENV_SCALE`) → generar mallas → generar dataset (reanudable) → empaquetar `dataset_B.npz`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/whiskit_dataset'; os.makedirs(WORK, exist_ok=True); print('Carpeta de trabajo:', WORK)

## 1. Compilar el simulador (con los archivos fuente ya parcheados)

In [ ]:
%%bash
set -e
apt-get install -y -q libboost-program-options-dev libboost-filesystem-dev libboost-system-dev freeglut3-dev libgl1-mesa-dev libglu1-mesa-dev > /dev/null
cd /content
if [ ! -d whiskitphysics ]; then git clone --depth 1 https://github.com/SeNSE-lab/whiskitphysics.git; fi
echo listo

In [ ]:
# Sobrescribimos los 4 archivos fuente con las versiones parcheadas (idénticas a las usadas en el piloto)
P='/content/whiskitphysics/code/'
open(P+'src/main.cpp','w').write(r'''/*
WHISKiT Physics Simulator
Copyright (C) 2019 Nadina Zweifel (SeNSE Lab)

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/>.

This code is based on code published by
Bullet Continuous Collision Detection and Physics Library
Copyright (c) 2015 Google Inc. http://bulletphysics.org

*/

#include "Simulation.h"
#include "CommonInterfaces/CommonExampleInterface.h"
#include "CommonInterfaces/CommonGUIHelperInterface.h"
#include "BulletCollision/CollisionDispatch/btCollisionObject.h"
#include "BulletCollision/CollisionShapes/btCollisionShape.h"
#include "BulletDynamics/Dynamics/btDiscreteDynamicsWorld.h"
#include "OpenGLWindow/SimpleOpenGL3App.h"
#include "Bullet3Common/b3Quaternion.h"

#include "LinearMath/btTransform.h"
#include "LinearMath/btHashMap.h"

#include <iostream>
#include <boost/program_options.hpp>

#include <signal.h>
#include <stdlib.h>
#include <string>

#include <boost/lexical_cast.hpp>
using boost::lexical_cast;

volatile sig_atomic_t exitFlag = 0;

void exit_function(int sigint)
{
	exitFlag = 1;
}

int main(int argc, char* argv[])
{
	signal(SIGINT, exit_function);
	Parameters* param = new Parameters();

  	try 
  	{ /** Define and parse the program options  */ 
	    namespace po = boost::program_options; 
	    po::options_description desc("Options");
	    desc.add_options() 
		("help,h", "Help screen")
		("DEBUG", po::value<int>(&param->DEBUG), "debug on/off")
		
		("TIME_STOP", po::value<float>(&param->TIME_STOP), "duration of simulation")

		("PRINT", po::value<int>(&param->PRINT), "print simulation output")
		("SAVE", po::value<int>(&param->SAVE), "saving on/off")
		("SAVE_VIDEO", po::value<int>(&param->SAVE_VIDEO), "video on/off")

		("OBJECT", po::value<int>(&param->OBJECT), "collision object ID (0: none, 1: stationary peg, 2: moving peg, 3: wall")
		
		("MODEL_TYPE", po::value<int>(&param->MODEL_TYPE), "model type: 0: average rat, 1: model Belli et al. 2018")
		("WHISKER_NAMES", po::value<std::vector<std::string> >(&param->WHISKER_NAMES)->multitoken(), "whisker names to simulate")
		("BLOW,b", po::value<float>(&param->BLOW), "whisker curvature on/off")
		("NO_MASS", po::value<int>(&param->NO_MASS), "whisker mass on/off")
		("NO_WHISKERS", po::value<int>(&param->NO_WHISKERS), "whisker on/off")

		("ACTIVE", po::value<int>(&param->ACTIVE), "active on/off")
		
		("POSITION", po::value<std::vector<std::string> >()->multitoken(), "initial position of rat")
		("ORIENTATION", po::value<std::vector<std::string> >()->multitoken(), "initial orientation of rat (euler angles)")
		
		("CDIST", po::value<float>(&param->CDIST), "distance of camera")
		("CPITCH", po::value<std::string>(), "head pitch")
		("CYAW", po::value<std::string>(), "head yaw")
		
		("SPEED", po::value<float>(&param->PEG_SPEED), "peg speed")
		("PEG_LOC", po::value<std::vector<std::string> >()->multitoken(), "peg location x y z (mm)")
		("ENV_LOC", po::value<std::vector<std::string> >()->multitoken(), "obj location x y z (mm)")
		("ENV_YAW", po::value<float>(&param->ENV_YAW), "obj yaw rotation (deg)")
		("ENV_SCALE", po::value<float>(&param->ENV_SCALE), "obj bounding diagonal (mm); 0 = file units")

		("dir_out", po::value<std::string>(&param->dir_out), "foldername for output file")
		("file_video", po::value<std::string>(&param->file_video), "filename of video")
		("file_env", po::value<std::string>(&param->file_env), "filename for environment");


	    po::variables_map vm; 


	    try { 
		    po::store(po::parse_command_line(argc, argv, desc,po::command_line_style::unix_style ^ po::command_line_style::allow_short), vm); // can throw 
		 	po::notify(vm);

		 	if ( vm.count("help")  ) { 
		        std::cout << "Bullet Whisker Simulation" << std::endl 
		                  << desc << std::endl; 
		        return 0; 
		    } 
			
			if (param->WHISKER_NAMES[0] == "ALL"){
	    		param->WHISKER_NAMES = {
	    			"LA0","LA1","LA2","LA3","LA4",
	    			"LB0","LB1","LB2","LB3","LB4",
	    			"LC0","LC1","LC2","LC3","LC4","LC5",
	    			"LD0","LD1","LD2","LD3","LD4","LD5",
	    			"LE1","LE2","LE3","LE4","LE5",
	    			"RA0","RA1","RA2","RA3","RA4",
	    			"RB0","RB1","RB2","RB3","RB4",
	    			"RC0","RC1","RC2","RC3","RC4","RC5",
	    			"RD0","RD1","RD2","RD3","RD4","RD5",
	    			"RE1","RE2","RE3","RE4","RE5"};
	    	}

	    	else if (param->WHISKER_NAMES[0] == "R"){
	    		param->WHISKER_NAMES = {
	    			
	    			"RA0","RA1","RA2","RA3","RA4",
	    			"RB0","RB1","RB2","RB3","RB4",
	    			"RC0","RC1","RC2","RC3","RC4","RC5",
	    			"RD0","RD1","RD2","RD3","RD4","RD5",
	    			"RE1","RE2","RE3","RE4","RE5"};
	    	}
			else if (param->WHISKER_NAMES[0] == "L"){
	    		param->WHISKER_NAMES = {
	    			"LA0","LA1","LA2","LA3","LA4",
	    			"LB0","LB1","LB2","LB3","LB4",
	    			"LC0","LC1","LC2","LC3","LC4","LC5",
	    			"LD0","LD1","LD2","LD3","LD4","LD5",
	    			"LE1","LE2","LE3","LE4","LE5"};
	    	}

			std::vector<std::string> coordinates;
			if (!vm["POSITION"].empty() && (coordinates = vm["POSITION"].as<std::vector<std::string> >()).size() == 3) {
				param->RATHEAD_LOC[0] = lexical_cast<float>(coordinates[0]);
				param->RATHEAD_LOC[1] = lexical_cast<float>(coordinates[1]);
				param->RATHEAD_LOC[2] = lexical_cast<float>(coordinates[2]);
			}	

			std::vector<std::string> pegloc;
			if (vm.count("PEG_LOC") && (pegloc = vm["PEG_LOC"].as<std::vector<std::string> >()).size() == 3) {
				param->PEG_LOC = btVector3(lexical_cast<float>(pegloc[0]), lexical_cast<float>(pegloc[1]), lexical_cast<float>(pegloc[2]));
			}

			std::vector<std::string> envloc;
			if (vm.count("ENV_LOC") && (envloc = vm["ENV_LOC"].as<std::vector<std::string> >()).size() == 3) {
				param->ENV_LOC = btVector3(lexical_cast<float>(envloc[0]), lexical_cast<float>(envloc[1]), lexical_cast<float>(envloc[2]));
			}

			std::vector<std::string> angles;
			if (!vm["ORIENTATION"].empty() && (angles = vm["ORIENTATION"].as<std::vector<std::string> >()).size() == 3) {
				param->RATHEAD_ORIENT[0] = lexical_cast<float>(angles[0]);
				param->RATHEAD_ORIENT[1] = lexical_cast<float>(angles[1]);
				param->RATHEAD_ORIENT[2] = lexical_cast<float>(angles[2]);
			}		

			
		  	DummyGUIHelper noGfx;

			CommonExampleOptions options(&noGfx);
			Simulation* simulation = new Simulation(options.m_guiHelper);

			// save parameters in simulation object
			simulation->parameters = param;
			simulation->initPhysics();
			std::cout.precision(17);
			
			// run simulation
			do{
				simulation->stepSimulation();
			}while(!(exitFlag || simulation->exitSim) );

			std::cout << "Saving data..." << std::endl;
			if(simulation->parameters->SAVE){
				std::cout << "Simualtion terminated." << std::endl;
				output* results = simulation->get_results();
				save_data(results,simulation->parameters->dir_out);
			}
			

			std::cout << "Exit simulation..." << std::endl;
			simulation->exitPhysics();

			delete simulation;
			std::cout << "Done." << std::endl;
			
	    } 
	    catch(po::error& e) 
	    { 
	      std::cerr << "ERROR: " << e.what() << std::endl << std::endl; 
	      std::cerr << desc << std::endl; 
	      return 1; 
	    } 
 
  	} 
  	catch(std::exception& e) 
  	{ 	std::cerr << "Unhandled Exception reached the top of main: "           << e.what() << ", application will now exit" << std::endl; 	return 2; 
 
  	} 

	
	return 0;
}
''')
open(P+'include/Parameters.h','w').write(r'''/*
WHISKiT Physics Simulator
Copyright (C) 2019 Nadina Zweifel (SeNSE Lab)

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/>.
*/

#ifndef PARAMETERS_H
#define PARAMETERS_H

#define PI 3.1415927

#define LINK_THETA_MAX PI/4
#define LINK_PHI_MAX PI/4
#define LINK_ZETA_MAX PI/360

#include <string>
#include <vector>
#include "LinearMath/btVector3.h"

class Parameters{
public:
	Parameters();
	~Parameters();
	
	// integration parameters
	float TIME_STEP;
	int NUM_STEP_INT;
	float TIME_STOP;

	// simulation parameters
	int DEBUG;
	int PRINT;
	int SAVE;
	int SAVE_VIDEO;
	
	// collision object type
	int OBJECT;
	// peg parameters
	btVector3 PEG_LOC;
	float PEG_SPEED; // in mm/second
	// curvWall parameters
	btVector3 curvWall_LOC;
	btVector3 curvWall_ORIENT;

	// whisker model parameters
	int MODEL_TYPE;
	std::string dir_param;
    std::vector<std::string> WHISKER_NAMES;
	std::vector<int> WHISKER_INDEX;
    float BLOW; // scale whisker diameter for visibility - ATTENTION: will affect dynamics!!!
	int NO_CURVATURE; // remove curvature for debugging
	int NO_MASS; 	  // set mass to zero for debugging
	int NO_WHISKERS;  // remove whiskers for debugging
	int NUM_LINKS;
	float RHO_BASE;
	float RHO_TIP;
	float E;
	float ZETA;
	
	// configuration parameters of rat head
	std::vector<float> RATHEAD_LOC;
	std::vector<float> RATHEAD_ORIENT;

	// whisking parameters
	int ACTIVE;
	std::string file_whisking_init_angle;
	std::string file_whisking_angle;

	// exploring
	int EXPLORING;
	std::vector<std::vector<float>> HEAD_LOC_VEL;
	std::vector<std::vector<float>> WHISKER_VEL;
	std::string dir_rathead;
	std::string dir_rathead_trajectory;


	// camera configuration
	btVector3 CPOS;
	float CDIST;
	float CPITCH;
	float CYAW;

	// directories/paths
	std::string dir_out;
	std::string file_video;
	std::string file_env;
	btVector3 ENV_LOC;      // posición del objeto .obj (mm)
	float ENV_YAW;          // rotación del objeto alrededor de z (grados)
	float ENV_SCALE;        // diagonal de la caja envolvente en mm (0 = usar unidades del archivo)
	std::string file_curvWall;
};

void set_default(Parameters* param);
std::vector<float> get_vector(float value, int N);
std::vector<float> stringToFloatVect(std::vector<std::string> vect_string);

#endif //PARAMETERS_H''')
open(P+'src/Parameters.cpp','w').write(r'''/*
WHISKiT Physics Simulator
Copyright (C) 2019 Nadina Zweifel (SeNSE Lab)

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/>.

*/

#include "Parameters.h"

// set default parameter values
Parameters::Parameters(){
	// input arguments for simulator
	DEBUG = 0;			// enable debug mode
	TIME_STEP = 0.001;	// set time step, this is related to output video's FPS
	NUM_STEP_INT = 100;	// set internal time step
	TIME_STOP = 1.5;		// set overall simulation time
    PRINT = 2;			// set to PRINT=1 to kinematics/dynamics realtime, set to PRINT = 2 to print simulation time
	SAVE = 0;			// save results to csv file
	SAVE_VIDEO = 0;		// save video when running main_opengl.cpp

	// collision object type
	OBJECT = 0;			// 0: nothing
						// 1: peg
						// 2: wall
						// 3: create object from 3D scan
				
	// parameters for peg (OBJECT = 1)
	PEG_LOC = btVector3(20, 25, 0);
	PEG_SPEED = 10;	

	// specify whisker configuration parameters
	MODEL_TYPE = 0; // Selects model type: 0 => average rat whisker array; 1 => model rat whisker array from Belli et al. 2018
	WHISKER_NAMES = {"RC0", "RC1", "RB1", "RD1", "LC0", "LC1", "LB1", "LD1"}; // select whiskers to simulate
	// WHISKER_NAMES = {"RA0", "RA1", "RA2", "RA3", "RA4", "RB0", "RB1", "RB2", "RB3", "RB4",
	// 				 "RC0", "RC1", "RC2", "RC3", "RC4", "RC5", "RC6", 
	// 				 "RD0", "RD1", "RD2", "RD3", "RD4", "RD5", "RD6", 
	// 				 "RE1", "RE2", "RE3", "RE4", "RE5", "RE6"};
    	
	BLOW = 1;				// increase whisker diameter for better visualization (will affect dynamics!!)
	NO_CURVATURE = 0;		// disable curvature
	NO_MASS = 0;			// disable mass of bodies
	NO_WHISKERS = 0;		// disable whiskers
	NUM_LINKS = 20;			// set number of links
	RHO_BASE = 1260.0;		// set density at whisker base
	RHO_TIP = 1690.0;		// set density at whisker tip
	E = 5e9;				// set young's modulus (GPa) at whisker base
	ZETA = 0.32;			// set damping coefficient zeta at whisker base

	// enable/disable whisking mode for added whiskers
	// Note: the whisking trajectory is pre-specified by user.
	ACTIVE = 1;				
	file_whisking_init_angle = ACTIVE?"whisking_init_angle.csv":"param_bp_angles.csv";
	file_whisking_angle = "whisking_trajectory.csv";

	// enable/disable exploring mode for rat head
	// Note: the head trajectory is 
	EXPLORING = 0;
	dir_rathead = "../data/object/NewRatHead.obj";
	dir_rathead_trajectory = "../data/rathead_trajectory_sample.csv";


	// rat position/orientation parameters
	RATHEAD_LOC = {0,0,0}; 			// set position of rathead
	RATHEAD_ORIENT = {0,0,0}; 		// set initial heading of rathead

	// camera parameters for visualization
	CPOS = btVector3(0, 26, 20);	// set camera pos relative to rathead
	CDIST=50;						// set camera distance
	CPITCH=-89;						// set camera pitch
	CYAW=0;							// set camera yaw

	// input/output file paths
	dir_out = "../output/test";
	file_video = "../output/video_test.mp4";
	file_env = "../data/environment/env_1.obj";
	ENV_LOC = btVector3(0,0,0);
	ENV_YAW = 0;
	ENV_SCALE = 1000.;   // comportamiento original: diagonal normalizada a 1000 mm	

}

// create a vector with same value
std::vector<float> get_vector(float value, int N){
	std::vector<float> vect;
	for(int i=0; i<N; i++){
		vect.push_back(value);
	}
	return vect;
}

// convert string to float vector - not used I think
std::vector<float> stringToFloatVect(std::vector<std::string> vect_string){
	std::string::size_type sz;
	std::vector<float> vect_float;
	for(int i=0; i<vect_string.size(); i++){
		vect_float.push_back(std::stof(vect_string[i],&sz));
	}
	return vect_float;
}

''')
open(P+'src/Simulation.cpp','w').write(r'''/*
WHISKiT Physics Simulator
Copyright (C) 2019 Nadina Zweifel (SeNSE Lab)

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/>.

This code is based on code published by
Bullet Continuous Collision Detection and Physics Library
Copyright (c) 2015 Google Inc. http://bulletphysics.org

*/


#include "Simulation.h"

btVector4 BLUE = btVector4(0.,0.,1.0,1);
btVector4 RED = btVector4(1.,0.,0.0,1);
btVector4 GREEN = btVector4(0.,1.,0.0,1);
btVector4 GREY = btVector4(0.,0.,0.0,0.5);
btVector4 YELLOW = btVector4(1.,1.,0.0,1);
btVector4 ORANGE = btVector4(1.,0.647,0.0,1);

void Simulation::stepSimulation(){
	auto start = std::chrono::high_resolution_clock::now(); 
	m_time += parameters->TIME_STEP; 								// increase time
	m_step += 1;													// increase step
	
	// run simulation as long as stop time not exceeded
	if(parameters->TIME_STOP==0 || m_time < parameters->TIME_STOP){

		// register collisions
		scabbers->detect_collision(m_dynamicsWorld);
		
		// first, push back data into data_dump 
		if(!parameters->NO_WHISKERS && parameters->SAVE){
			scabbers->dump_M(data_dump);
			scabbers->dump_F(data_dump);
			scabbers->dump_Q(data_dump);
		}

		// moving object 1
		if(parameters->OBJECT==1){
			if(parameters->PEG_SPEED>0){
				btVector3 velocity = parameters->PEG_SPEED * btVector3(0.4,-1,0).normalized();
				peg->setLinearVelocity(velocity);
				
			}
			
		}

		// move array if in ACTIVE mode
		if(parameters->ACTIVE && !parameters->NO_WHISKERS){
			scabbers->whisk(m_step, parameters->WHISKER_VEL);
			
		}
		
		// move rat head if in EXPLORING mode
		if(parameters->EXPLORING){
			this_loc_vel = parameters->HEAD_LOC_VEL[m_step-1];
			scabbers->setLinearVelocity(btVector3(this_loc_vel[3], this_loc_vel[4], this_loc_vel[5]/10));
			// scabbers->setLinearVelocity(btVector3(0, 0, 0));
			scabbers->setAngularVelocity(btVector3(this_loc_vel[6], this_loc_vel[7], this_loc_vel[8]));
			// scabbers->setAngularVelocity(btVector3(0, 0, 0));
		}

		// step simulation
		m_dynamicsWorld->stepSimulation(parameters->TIME_STEP,parameters->NUM_STEP_INT,parameters->TIME_STEP/parameters->NUM_STEP_INT);

		// draw debug if enabled
	    if(parameters->DEBUG){
	    	m_dynamicsWorld->debugDrawWorld();
	    }

	    // set exit flag to zero
	    exitSim = 0;
	}
	else{
		// timeout -> set exit flg
		exitSim = 1;
	}
	
	auto stop = std::chrono::high_resolution_clock::now(); 
	auto duration = std::chrono::duration_cast<std::chrono::milliseconds>(stop - start);
	m_time_elapsed += duration.count()/1000.f;
	auto factor = m_time_elapsed / m_time;
	auto time_remaining = (int)((parameters->TIME_STOP - m_time) * (factor));
	if(parameters->PRINT==2){
		std::cout << "\rSimulation time: " << std::setprecision(2) << m_time << "s\tCompleted: " << std::setprecision(2) << m_time/parameters->TIME_STOP*100 << " %\tTime remaining: " << std::setprecision(4) << time_remaining/60 << " min " << std::setprecision(4) << (time_remaining % 60) << " s\n" << std::flush;
	}
    
}

void Simulation::initPhysics()
{	
	vec = btVector3(0.5,-1,0).normalized();
	data_dump->init(parameters->WHISKER_NAMES);

	// set visual axis
	m_guiHelper->setUpAxis(2);

	// create empty dynamics world[0]
	m_collisionConfiguration = new btDefaultCollisionConfiguration(); 
    m_dispatcher = new	btCollisionDispatcher(m_collisionConfiguration); 

    // broadphase algorithm
    m_broadphase = new btDbvtBroadphase();

	// select solver
	std::cout << "Using btSequentialImpulseConstraintSolver..." << std::endl;
	m_solver = new btSequentialImpulseConstraintSolver();

	m_dynamicsWorld = new btDiscreteDynamicsWorld(m_dispatcher,m_broadphase,m_solver,m_collisionConfiguration);			

	// set number of iterations
	m_dynamicsWorld->getSolverInfo().m_numIterations = 20;
	m_dynamicsWorld->getSolverInfo().m_solverMode = SOLVER_SIMD |
                        SOLVER_USE_WARMSTARTING |
                        SOLVER_RANDMIZE_ORDER |
                        0;
	m_dynamicsWorld->getSolverInfo().m_splitImpulse = true;
	m_dynamicsWorld->getSolverInfo().m_erp = 0.8f;

	
	// set gravity
	m_dynamicsWorld->setGravity(btVector3(0,0,0));

    // create debug drawer
	m_guiHelper->createPhysicsDebugDrawer(m_dynamicsWorld);

	if (m_dynamicsWorld->getDebugDrawer()){
		if(parameters->DEBUG==1){
			std::cout << "DEBUG option 1: wireframes." << std::endl;
			m_dynamicsWorld->getDebugDrawer()->setDebugMode(btIDebugDraw::DBG_DrawWireframe);
		}
		else if(parameters->DEBUG==2){
			std::cout << "DEBUG option 2: constraints." << std::endl;
			m_dynamicsWorld->getDebugDrawer()->setDebugMode(btIDebugDraw::DBG_DrawConstraints);
		}
		else if(parameters->DEBUG==3){
			std::cout << "DEBUG option 3: wireframes & constraints." << std::endl;
			m_dynamicsWorld->getDebugDrawer()->setDebugMode(btIDebugDraw::DBG_DrawWireframe+btIDebugDraw::DBG_DrawConstraintLimits);
		}
		else if(parameters->DEBUG==4){
			std::cout << "DEBUG option 4: AAbb." << std::endl;
			m_dynamicsWorld->getDebugDrawer()->setDebugMode(btIDebugDraw::DBG_DrawAabb);
		}
		else if(parameters->DEBUG==5){
			std::cout << "DEBUG option 5: Frammes." << std::endl;
			m_dynamicsWorld->getDebugDrawer()->setDebugMode(btIDebugDraw::DBG_DrawFrames);
		}
		else if(parameters->DEBUG==6){
			std::cout << "DEBUG option 6: Only collision" << std::endl;
			// m_dynamicsWorld->getDebugDrawer()->setDebugMode(btIDebugDraw::DBG_DrawFrames);
		}
		else{
			std::cout << "No DEBUG." << std::endl;
			m_dynamicsWorld->getDebugDrawer()->setDebugMode(btIDebugDraw::DBG_NoDebug);
		}
	}

	// add rat to world
	scabbers = new Rat(m_guiHelper,m_dynamicsWorld, &m_collisionShapes, parameters);
	btVector3 rathead_pos = scabbers->getPosition();
	
	// create object to collide with
	// peg
	if(parameters->OBJECT==1){
		btCollisionShape* pegShape = new btCylinderShapeZ(btVector3(1,1,80));
		pegShape->setMargin(0.1);
		m_collisionShapes.push_back(pegShape);
		btTransform trans = createFrame(parameters->PEG_LOC,btVector3(0, 0, 0));
		peg = createDynamicBody(1,0.5,trans, pegShape, m_guiHelper,  BLUE);
		m_dynamicsWorld->addRigidBody(peg,COL_ENV,envCollidesWith);
		peg->setActivationState(DISABLE_DEACTIVATION);
		
	}
	// create object to collide with wall
	else if(parameters->OBJECT==2){
		btCollisionShape* wallShape = new btBoxShape(btVector3(5,200,60));
		wallShape->setMargin(0.1);
		m_collisionShapes.push_back(wallShape);
		btTransform trans = createFrame(btVector3(50,0,0),btVector3(0,0,PI/6));
		wall = createDynamicBody(0,0.5, trans, wallShape, m_guiHelper,  BLUE);
		m_dynamicsWorld->addRigidBody(wall,COL_ENV,envCollidesWith);
	}
	// create object from 3D scan
	else if(parameters->OBJECT==3){
		// add environment to world
		btVector4 envColor = btVector4(0.6,0.6,0.6,1);
		btTransform envTrans = createFrame(parameters->ENV_LOC, btVector3(0,0,parameters->ENV_YAW*PI/180.));
		env = new Object(m_guiHelper,m_dynamicsWorld, &m_collisionShapes,envTrans,parameters->file_env,envColor,btScalar(parameters->ENV_SCALE),btScalar(0),COL_ENV,envCollidesWith);
	}
	
	// generate graphics
	m_guiHelper->autogenerateGraphicsObjects(m_dynamicsWorld);

	// set camera position to rat head
	camPos[0] = rathead_pos[0]+parameters->CPOS[0];
	camPos[1] = rathead_pos[1]+parameters->CPOS[1];
	camPos[2] = rathead_pos[2]+parameters->CPOS[2];
	camDist = parameters->CDIST;
	camPitch = parameters->CPITCH;
	camYaw = parameters->CYAW;
	resetCamera();

	// if active whisking, load whisking protraction angle trajectory
	if (parameters->ACTIVE){
		read_csv_float(parameters->dir_param + parameters->file_whisking_angle, parameters->WHISKER_VEL);
		parameters->TIME_STOP = std::min(parameters->TIME_STOP, (parameters->WHISKER_VEL[0].size()/3 - 1) * parameters->TIME_STEP);
	}

	// if exploring, load data for rat head trajectory
	if (parameters->EXPLORING){
		read_csv_float(parameters->dir_rathead_trajectory, parameters->HEAD_LOC_VEL);
	}

	

	// initialize time/step tracker
	m_time_elapsed = 0;
	m_time = 0;
	m_step = 0;
	
	std::cout << "\n\nStart simulation..." << std::endl;
	std::cout << "\n====================================================\n" << std::endl;
}

output* Simulation::get_results(){
	return data_dump;
}

void Simulation::resetCamera(){	
	m_guiHelper->resetCamera(camDist,camYaw,camPitch,camPos[0],camPos[1],camPos[2]);
}''')
print('fuentes parcheadas')

In [ ]:
%%bash
set -e
cd /content/whiskitphysics/code && mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release > /dev/null 2>&1
make -j$(nproc) whiskit 2>&1 | grep -E "error|Built target whiskit"
./whiskit --help | grep -E "ENV_LOC|PEG_LOC" && echo "COMPILADO OK" 

## 2. Scripts, mallas y geometría de los bigotes

In [ ]:
import os
os.makedirs('/content/dataset', exist_ok=True)
open('/content/dataset/make_shapes.py','w').write(r'''#!/usr/bin/env python3
"""
Genera mallas .obj (triángulos) para la Tarea B. Unidades: milímetros. Cada malla está centrada en (0,0)
en el plano x-y y se extiende en z de -H/2 a +H/2 (los bigotes baten aproximadamente en el plano z≈0).

Formas:
  cylinder_r{R}.obj   cilindro vertical de radio R (como el poste, pero con radio variable)
  prism_{A}x{B}.obj   prisma rectangular de lados A (en x) y B (en y); al rotarlo con ENV_YAW cambia la cara que tocan
  wave_a{A}_l{L}.obj  pared ondulada: superficie x = A·sin(2π y / L), extruida en z (contorno no convexo)

Por qué triángulos explícitos: WHISKiT construye la malla de colisión a partir de la lista de vértices de 3 en 3,
así que escribimos cada triángulo con sus 3 vértices (sin compartir índices) para que la colisión sea exacta.
"""
import argparse, math, os
import numpy as np

H = 60.0  # altura (mm) suficiente para cubrir el plano de batido


def write_obj(path, tris):
    """tris: lista de triángulos, cada uno 3 puntos (x,y,z). Se escriben vértices sin compartir."""
    with open(path, "w") as f:
        f.write("# generado por make_shapes.py (mm)\n")
        for t in tris:
            for p in t:
                f.write(f"v {p[0]:.4f} {p[1]:.4f} {p[2]:.4f}\n")
        for i in range(len(tris)):
            b = 3 * i + 1
            f.write(f"f {b} {b+1} {b+2}\n")


def extrude_polyline(pts, closed, h=H):
    """Convierte una polilínea 2D en una banda vertical de triángulos (dos por segmento)."""
    tris = []
    n = len(pts)
    rng = range(n) if closed else range(n - 1)
    for i in rng:
        a, b = pts[i], pts[(i + 1) % n]
        a0, a1 = (a[0], a[1], -h / 2), (a[0], a[1], h / 2)
        b0, b1 = (b[0], b[1], -h / 2), (b[0], b[1], h / 2)
        tris.append((a0, b0, b1)); tris.append((a0, b1, a1))
    return tris


def cylinder(r, n=48):
    pts = [(r * math.cos(2 * math.pi * k / n), r * math.sin(2 * math.pi * k / n)) for k in range(n)]
    return extrude_polyline(pts, closed=True)


def prism(a, b):
    pts = [(-a / 2, -b / 2), (a / 2, -b / 2), (a / 2, b / 2), (-a / 2, b / 2)]
    return extrude_polyline(pts, closed=True)


def wave(amp, lam, length=30.0, n=60, thickness=5.0):
    """Pared cuyo frente sigue x = amp*sin(2π y/λ); cerrada por detrás para que sea un sólido."""
    ys = np.linspace(-length / 2, length / 2, n)
    front = [(amp * math.sin(2 * math.pi * y / lam), y) for y in ys]
    back = [(x - thickness - amp, y) for x, y in front][::-1]
    return extrude_polyline(front + back, closed=True)


def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--out", required=True); a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    specs = {}
    for r in [1.0, 2.5, 5.0]:
        specs[f"cylinder_r{r:g}.obj"] = cylinder(r)
    for A, B in [(4, 12), (8, 8), (6, 20)]:
        specs[f"prism_{A}x{B}.obj"] = prism(A, B)
    for amp, lam in [(2, 15), (4, 25), (3, 10)]:
        specs[f"wave_a{amp}_l{lam}.obj"] = wave(amp, lam)
    for name, tris in specs.items():
        write_obj(os.path.join(a.out, name), tris)
        print(f"{name:22s} {len(tris):4d} triángulos")


if __name__ == "__main__":
    main()
''')
open('/content/dataset/run_dataset_B.py','w').write(r'''#!/usr/bin/env python3
"""
Generador de dataset para la Tarea B: clasificar la FORMA del objeto y estimar su POSE a partir del torque.

Cada simulación: se elige al azar una malla (3 familias × 3 tamaños), una posición (x, y) y una rotación (yaw).
Etiquetas en el manifiesto:  shape_family (cylinder / prism / wave), mesh (archivo), x, y, yaw, y el parámetro
de tamaño de la familia (radio, lados, amplitud/longitud de onda).

Uso:
   python3 run_dataset_B.py --n 18 --seed 0 --out pilot_B --time 1.0 --jobs 2
Requiere las mallas en whiskitphysics/code/data/shapes (python3 make_shapes.py --out .../data/shapes)
"""
import argparse, csv, os, subprocess, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
CODE = os.path.abspath(os.path.join(HERE, "..", "whiskitphysics", "code"))
BIN = os.path.join(CODE, "build", "whiskit")
SCRIPTS = os.path.join(CODE, "scripts")

MESHES = {  # archivo -> (familia, parámetro de tamaño)
    "cylinder_r1.obj": ("cylinder", "r=1"), "cylinder_r2.5.obj": ("cylinder", "r=2.5"), "cylinder_r5.obj": ("cylinder", "r=5"),
    "prism_4x12.obj": ("prism", "4x12"), "prism_8x8.obj": ("prism", "8x8"), "prism_6x20.obj": ("prism", "6x20"),
    "wave_a2_l15.obj": ("wave", "a2_l15"), "wave_a4_l25.obj": ("wave", "a4_l25"), "wave_a3_l10.obj": ("wave", "a3_l10"),
}
# Pose: el objeto se coloca delante/al lado derecho de la cabeza. x más lejos que en A para reducir la
# penetración inicial de objetos grandes (los bigotes en reposo llegan a x≈36).
X_RANGE, Y_RANGE, YAW_RANGE = (18.0, 46.0), (6.0, 32.0), (0.0, 180.0)


# ---- Comprobación geométrica de la pose -------------------------------------------------------
# Rechazamos poses en las que el objeto (a) atraviesa algún bigote en reposo (t=0) -> artefacto de
# penetración con torques enormes, o (b) no es alcanzado por ningún bigote durante un ciclo de batido
# -> muestra sin información. Usa la huella 2D del objeto y las coordenadas de los bigotes guardadas
# en whisker_geometry.npz (extraídas de una simulación de batido libre).
import sys; sys.path.insert(0, HERE)
from make_shapes import cylinder, prism, wave
_GEO = np.load(os.path.join(HERE, "whisker_geometry.npz"))
_FOOT = {  # polígono (x,y) en el marco del objeto, obtenido de la misma función que genera la malla
    "cylinder_r1.obj": cylinder(1.0), "cylinder_r2.5.obj": cylinder(2.5), "cylinder_r5.obj": cylinder(5.0),
    "prism_4x12.obj": prism(4, 12), "prism_8x8.obj": prism(8, 8), "prism_6x20.obj": prism(6, 20),
    "wave_a2_l15.obj": wave(2, 15), "wave_a4_l25.obj": wave(4, 25), "wave_a3_l10.obj": wave(3, 10),
}

def footprint(mesh):
    tris = _FOOT[mesh]                      # banda vertical: el primer vértice de cada triángulo par recorre el contorno
    return np.array([[t[0][0], t[0][1]] for t in tris[::2]], dtype=np.float32)

def inside(poly, pts):
    """Test punto-en-polígono (ray casting) vectorizado. poly: (M,2), pts: (K,2) -> (K,) bool"""
    x, y = pts[:, 0], pts[:, 1]; res = np.zeros(len(pts), bool)
    j = len(poly) - 1
    for i in range(len(poly)):
        xi, yi = poly[i]; xj, yj = poly[j]
        cond = ((yi > y) != (yj > y)) & (x < (xj - xi) * (y - yi) / (yj - yi + 1e-12) + xi)
        res ^= cond; j = i
    return res

def pose_ok(mesh, x, y, yaw, margin=1.5):
    poly = footprint(mesh); c, s_ = np.cos(np.radians(yaw)), np.sin(np.radians(yaw))
    R = np.array([[c, -s_], [s_, c]], dtype=np.float32)
    world = poly @ R.T + np.array([x, y], dtype=np.float32)
    # (a) sin penetración en reposo: inflamos el polígono con un margen alrededor de su centro
    ctr = world.mean(0); infl = ctr + (world - ctr) * (1 + margin / max(np.abs(world - ctr).max(), 1e-6))
    if inside(infl, _GEO["rest"]).any():
        return False
    # (b) alcanzable: algún punto del barrido cae dentro del objeto (habrá contacto)
    return inside(world, _GEO["sweep"]).any()


def sample(n, seed):
    rng = np.random.default_rng(seed); names = sorted(MESHES)
    out, tries = [], 0
    while len(out) < n:
        mesh = names[len(out) % len(names)]                 # balance entre mallas
        x, y, yaw = float(rng.uniform(*X_RANGE)), float(rng.uniform(*Y_RANGE)), float(rng.uniform(*YAW_RANGE))
        tries += 1
        if pose_ok(mesh, x, y, yaw):
            out.append(dict(mesh=mesh, x=x, y=y, yaw=yaw))
    print(f"[sample] {n} poses válidas de {tries} intentos", flush=True)
    rng.shuffle(out)
    return out


def run_one(i, s, out_dir, t_stop):
    d = os.path.join(out_dir, f"sim_{i:04d}"); flag = os.path.join(d, "DONE")
    if os.path.exists(flag):
        return i, "skip", 0.0
    os.makedirs(d, exist_ok=True)
    cmd = [BIN, "--MODEL_TYPE", "0", "--DEBUG", "0", "--PRINT", "0", "--OBJECT", "3",
           "--file_env", f"../data/shapes/{s['mesh']}", "--ENV_SCALE", "0",
           "--ENV_LOC", f"{s['x']:.3f}", f"{s['y']:.3f}", "0", "--ENV_YAW", f"{s['yaw']:.2f}",
           "--TIME_STOP", f"{t_stop}", "--ACTIVE", "1", "--WHISKER_NAMES", "R",
           "--SAVE", "1", "--SAVE_VIDEO", "0", "--dir_out", d]
    t0 = time.time()
    with open(os.path.join(d, "whiskit.log"), "w") as log:
        rc = subprocess.call(cmd, cwd=SCRIPTS, stdout=log, stderr=subprocess.STDOUT)
    dt = time.time() - t0
    if rc == 0 and os.path.exists(os.path.join(d, "dynamics", "Mz.csv")):
        open(flag, "w").write(f"{dt:.1f}\n"); return i, "ok", dt
    return i, f"fail(rc={rc})", dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=18); ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="pilot_B"); ap.add_argument("--time", type=float, default=1.0)
    ap.add_argument("--jobs", type=int, default=2); a = ap.parse_args()
    out_dir = os.path.join(HERE, a.out); os.makedirs(out_dir, exist_ok=True)
    samples = sample(a.n, a.seed)
    with open(os.path.join(out_dir, "manifest.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["sim_id", "shape_family", "size_param", "mesh", "x_mm", "y_mm", "yaw_deg", "t_stop_s", "seed"])
        for i, s in enumerate(samples):
            fam, size = MESHES[s["mesh"]]
            w.writerow([i, fam, size, s["mesh"], f"{s['x']:.3f}", f"{s['y']:.3f}", f"{s['yaw']:.2f}", a.time, a.seed])
    print(f"[dataset B] {a.n} simulaciones -> {out_dir} (jobs={a.jobs}, t={a.time}s)", flush=True)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=a.jobs) as ex:
        futs = [ex.submit(run_one, i, s, out_dir, a.time) for i, s in enumerate(samples)]
        for k, fu in enumerate(futs, 1):
            i, st, dt = fu.result()
            print(f"[{k:3d}/{a.n}] sim_{i:04d} {st:10s} {dt:6.1f}s  (transcurrido {(time.time()-t0)/60:5.1f} min)", flush=True)
    print("[dataset B] listo", flush=True)


if __name__ == "__main__":
    main()
''')
open('/content/dataset/build_npz_B.py','w').write(r'''#!/usr/bin/env python3
"""
Empaqueta un dataset de la Tarea B. Igual que build_npz.py, pero con etiquetas de forma y pose:
   X        (N, T, 27, 6)
   y_class  (N,)   0=cylinder, 1=prism, 2=wave
   y_pose   (N, 3) x, y (mm), yaw (grados)
   mesh     (N,)   nombre de la malla (para análisis por tamaño)
"""
import argparse, csv, json, os
import numpy as np, pandas as pd

CHANNELS = ["Mx", "My", "Mz", "Fx", "Fy", "Fz"]; CLASSES = ["cylinder", "prism", "wave"]


def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--dir", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--drop", type=int, default=5); a = ap.parse_args()
    base = os.path.join(os.path.dirname(os.path.abspath(__file__)), a.dir)
    man = list(csv.DictReader(open(os.path.join(base, "manifest.csv"))))
    X, yc, yp, mesh, ids, dropped, names, T = [], [], [], [], [], [], None, None
    for r in man:
        sid = int(r["sim_id"]); d = os.path.join(base, f"sim_{sid:04d}")
        if not os.path.exists(os.path.join(d, "DONE")):
            dropped.append((sid, "no terminada")); continue
        if names is None:
            names = [s.strip() for s in open(os.path.join(d, "whisker_ID.csv")).read().split(",") if s.strip()]
        arr = np.stack([pd.read_csv(os.path.join(d, "dynamics", f"{c}.csv"), header=None).values[:, :len(names)] for c in CHANNELS], -1)[a.drop:].astype(np.float32)
        if np.isnan(arr).any() or np.abs(arr).max() > 1e6:
            dropped.append((sid, "NaN/explosión")); continue
        T = arr.shape[0] if T is None else min(T, arr.shape[0])
        X.append(arr); yc.append(CLASSES.index(r["shape_family"])); yp.append([float(r["x_mm"]), float(r["y_mm"]), float(r["yaw_deg"])])
        mesh.append(r["mesh"]); ids.append(sid)
    X = np.stack([x[:T] for x in X]).astype(np.float32)
    np.savez_compressed(os.path.join(os.path.dirname(base), a.out), X=X, y_class=np.array(yc), y_pose=np.array(yp, np.float32),
                        mesh=np.array(mesh), sim_id=np.array(ids), whisker_names=np.array(names), channel_names=np.array(CHANNELS), classes=np.array(CLASSES))
    print(f"X {X.shape}  clases {np.bincount(yc, minlength=3)}  guardado en {a.out}")
    if dropped: print("excluidas:", dropped)


if __name__ == "__main__":
    main()
''')
print('scripts escritos')

In [ ]:
# Geometría de reposo/barrido de los bigotes (para la comprobación de poses): se extrae de una simulación corta sin objeto.
%cd /content/whiskitphysics/code/scripts
!mkdir -p ../output/free && ../build/whiskit --MODEL_TYPE 0 --DEBUG 0 --PRINT 0 --OBJECT 0 --TIME_STOP 0.13 --ACTIVE 1 --WHISKER_NAMES R --SAVE 1 --SAVE_VIDEO 0 --dir_out ../output/free > /dev/null
import numpy as np, pandas as pd, os
os.chdir('/content/whiskitphysics/code/output/free')
w=[s.strip() for s in open('whisker_ID.csv').read().split(',') if s.strip()]
def pts_at(ts):
    out=[]
    for n in w:
        x=pd.read_csv(f'kinematics/x/{n}.csv',header=None).values; y=pd.read_csv(f'kinematics/y/{n}.csv',header=None).values
        for t in ts:
            xy=np.stack([x[t],y[t]],1); out.append(xy[~np.isnan(xy).any(1)])
    return np.concatenate(out).astype(np.float32)
np.savez('/content/dataset/whisker_geometry.npz', rest=pts_at([0]), sweep=pts_at(range(0,125,5)))
!cd /content/dataset && python3 make_shapes.py --out /content/whiskitphysics/code/data/shapes
%cd /content

In [ ]:
import os
link='/content/dataset/full_B'
if not os.path.islink(link):
    os.makedirs(f'{WORK}/full_B', exist_ok=True); os.symlink(f'{WORK}/full_B', link)
print('salida ->', os.path.realpath(link))

## 3. Generar (reanudable). 9 mallas × ~33 poses = 300 simulaciones de 1 s

In [ ]:
import os
!cd /content/dataset && python3 run_dataset_B.py --n 300 --seed 1 --out full_B --time 1.0 --jobs {os.cpu_count()}

## 4. Empaquetar y guardar en Drive

In [ ]:
!cd /content/dataset && python3 build_npz_B.py --dir full_B --out dataset_B.npz
import shutil, numpy as np; shutil.copy('/content/dataset/dataset_B.npz', f'{WORK}/dataset_B.npz')
d=np.load(f'{WORK}/dataset_B.npz'); print('X', d['X'].shape, 'clases', np.bincount(d['y_class']))